# Thai Word Segmentation
**NLP Hackathon — Character-level BIO Tagging**

Task: Predict B_WORD / I_WORD / E_WORD for each non-whitespace character in the test set.  
Metric: Macro F1-score

## Strategy
1. **Multi-engine tokenizer ensemble** (PyThaiNLP `newmm` + `longest`) to segment Thai text
2. Convert word segmentation → character-level B/I/E labels
3. **Character feature engineering + ML classifier** (LightGBM) trained on pseudo-labeled data
4. Confidence-weighted ensemble of all approaches for final prediction

In [2]:
!pip install pythainlp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 51.4 MB/s eta 0:00:00


In [3]:
# ── 0. Imports & Setup ─────────────────────────────────────────────────────────
import re
import unicodedata
import numpy as np
import pandas as pd
from collections import Counter
from pathlib import Path
from google.colab import files
import os


from pythainlp.tokenize import word_tokenize
from pythainlp.util import is_thai_char

import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import StratifiedKFold

import warnings
warnings.filterwarnings('ignore')


# file "https://www.kaggle.com/settings"
files.upload() #upload kaggle.json (Legacy API Credentials)

!mkdir -p ~/.kaggle/

# Move the uploaded kaggle.json file to the .kaggle directory
!mv kaggle.json ~/.kaggle/

# Set secure permissions for the API key file (read-only for owner)
!chmod 600 ~/.kaggle/kaggle.json

!kaggle competitions download -c super-ai-engineer-ss-6-word-segmentation

!unzip super-ai-engineer-ss-6-word-segmentation.zip

# Paths
BASE_DIR = Path('.')
TEST_FILE = BASE_DIR / 'ws_test.txt'
SAMPLE_SUB = BASE_DIR / 'ws_sample_submission.csv'
OUTPUT_FILE = BASE_DIR / 'ws_submission.csv'

print('✅ Imports ready')
print(f'   pythainlp version: {__import__("pythainlp").__version__}')

Saving kaggle.json to kaggle.json
100% 1.95M/1.95M [00:00<00:00, 149MB/s]

Archive:  super-ai-engineer-ss-6-word-segmentation.zip
  inflating: LST20 Annotation Guideline.pdf  
  inflating: LST20 Brief Specification.pdf  
  inflating: ws_list.txt             
  inflating: ws_sample_submission.csv  
  inflating: ws_test.txt             
✅ Imports ready
   pythainlp version: 5.3.4


In [4]:
# ── 1. Load & Inspect Test Data ────────────────────────────────────────────────
with open(TEST_FILE, 'r', encoding='utf-8') as f:
    test_text = f.read()

print(f'Total characters (incl. whitespace): {len(test_text):,}')

# Build character index mapping: id (1-indexed) -> char
char_ids = []        # 1-indexed positions of ALL chars
non_ws_ids = []      # 1-indexed positions of non-whitespace chars only
non_ws_chars = []    # the actual non-ws characters

for i, c in enumerate(test_text):
    idx = i + 1  # 1-indexed
    char_ids.append(idx)
    if c != ' ':  # whitespace excluded from submission
        non_ws_ids.append(idx)
        non_ws_chars.append(c)

print(f'Non-whitespace chars: {len(non_ws_chars):,}')
print(f'Expected submission rows: {len(non_ws_ids):,}')

# Verify against sample submission
sample_df = pd.read_csv(SAMPLE_SUB)
print(f'Sample submission rows: {len(sample_df):,}')
assert len(non_ws_ids) == len(sample_df), '⚠️  Mismatch in row count!'
assert list(non_ws_ids) == list(sample_df['Id']), '⚠️  ID mismatch!'
print('✅ IDs verified against sample submission')

# Show first 60 chars with IDs
print('\nFirst 20 non-whitespace chars:')
for i in range(20):
    print(f'  id={non_ws_ids[i]:3d}  char={repr(non_ws_chars[i])}')

Total characters (incl. whitespace): 37,248
Non-whitespace chars: 35,182
Expected submission rows: 35,182
Sample submission rows: 35,182
✅ IDs verified against sample submission

First 20 non-whitespace chars:
  id=  1  char='ท'
  id=  2  char='ี'
  id=  3  char='่'
  id=  4  char='ย'
  id=  5  char='ั'
  id=  6  char='ง'
  id=  7  char='ส'
  id=  8  char='ถ'
  id=  9  char='า'
  id= 10  char='น'
  id= 11  char='ก'
  id= 12  char='า'
  id= 13  char='ร'
  id= 14  char='ณ'
  id= 15  char='์'
  id= 16  char='ย'
  id= 17  char='ั'
  id= 18  char='ง'
  id= 19  char='ไ'
  id= 20  char='ม'


In [5]:
# ── 2. Helper: Words → B/I/E Labels ────────────────────────────────────────────
def words_to_bio(words):
    """
    Convert a list of word strings to character-level B/I/E labels.
    Single-character words → B_WORD only (treated as B_WORD + implicit E).
    Per the task spec: single-char word = B_WORD label.
    2-char word = [B_WORD, E_WORD]
    3+ char word = [B_WORD, I_WORD, ..., E_WORD]
    """
    labels = []
    for word in words:
        n = len(word)
        if n == 1:
            labels.append('B_WORD')
        elif n == 2:
            labels.extend(['B_WORD', 'E_WORD'])
        else:
            labels.append('B_WORD')
            labels.extend(['I_WORD'] * (n - 2))
            labels.append('E_WORD')
    return labels


def text_to_bio_labels(text, engine='newmm'):
    """
    Tokenize text with PyThaiNLP and compute B/I/E labels
    for each non-whitespace character, keeping whitespace positions.
    Returns: (non_ws_chars, non_ws_bio_labels)
    """
    # Tokenize keeping whitespace boundaries intact
    words = word_tokenize(text, engine=engine, keep_whitespace=True)

    all_labels = []  # labels for every character (incl. whitespace)
    for word in words:
        if word == ' ':
            all_labels.append('SPACE')
        else:
            all_labels.extend(words_to_bio([word]))

    # Sanity: char count must match
    reconstructed = ''.join(words)
    if len(reconstructed) != len(all_labels):
        raise ValueError(f'Label count mismatch: {len(reconstructed)} chars vs {len(all_labels)} labels')

    # Filter out whitespace
    chars_out, labels_out = [], []
    for c, l in zip(reconstructed, all_labels):
        if l != 'SPACE':
            chars_out.append(c)
            labels_out.append(l)

    return chars_out, labels_out


# Quick test
sample = 'สถานการณ์ยังไม่คลี่คลาย'
c_test, l_test = text_to_bio_labels(sample, engine='newmm')
for c, l in zip(c_test, l_test):
    print(f'  {repr(c):4s} → {l}')

  'ส'  → B_WORD
  'ถ'  → I_WORD
  'า'  → I_WORD
  'น'  → I_WORD
  'ก'  → I_WORD
  'า'  → I_WORD
  'ร'  → I_WORD
  'ณ'  → I_WORD
  '์'  → E_WORD
  'ย'  → B_WORD
  'ั'  → I_WORD
  'ง'  → E_WORD
  'ไ'  → B_WORD
  'ม'  → I_WORD
  '่'  → E_WORD
  'ค'  → B_WORD
  'ล'  → I_WORD
  'ี'  → I_WORD
  '่'  → I_WORD
  'ค'  → I_WORD
  'ล'  → I_WORD
  'า'  → I_WORD
  'ย'  → E_WORD


In [6]:
# ── 3. Character Feature Engineering ──────────────────────────────────────────

# Thai character sets
THAI_CONSONANTS = set('กขฃคฅฆงจฉชซฌญฎฏฐฑฒณดตถทธนบปผฝพฟภมยรลวศษสหฬอฮ')
THAI_LEAD_VOWELS = set('เแโใไ')   # vowels that appear before consonant
THAI_FOLL_VOWELS = set('ะัาิีึืุูํ')  # vowels that follow consonant
THAI_TONEMARKS   = set('่้๊๋')
THAI_ABOVE_BELOW = set('็ิีึืุูํ์ๆ')  # above/below diacritics
THAI_DIGITS      = set('๐๑๒๓๔๕๖๗๘๙')
ALL_THAI = THAI_CONSONANTS | THAI_LEAD_VOWELS | THAI_FOLL_VOWELS | THAI_TONEMARKS | THAI_DIGITS


def get_char_type(c):
    """Return integer type code for a character."""
    if c in THAI_CONSONANTS:  return 0
    if c in THAI_LEAD_VOWELS: return 1
    if c in THAI_FOLL_VOWELS: return 2
    if c in THAI_TONEMARKS:   return 3
    if c in THAI_ABOVE_BELOW: return 4
    if c in THAI_DIGITS:      return 5
    if c.isdigit():           return 6
    if c.isalpha():           return 7
    if c in ' '  :           return 8
    return 9  # punctuation / other


def is_word_boundary_hint(chars, i):
    """Heuristic: is position i likely a word boundary?"""
    c = chars[i]
    prev_c = chars[i - 1] if i > 0 else ''
    next_c = chars[i + 1] if i < len(chars) - 1 else ''

    # Transition from non-Thai to Thai or vice versa → boundary
    if prev_c and (is_thai_char(c) != is_thai_char(prev_c)):
        return 1
    # Digit or punct → likely start of new token
    if c in '.,!?;:()[]{}"' or prev_c in '.,!?;:()[]{}"':
        return 1
    return 0


WINDOW = 5  # context window size on each side

def extract_features(chars):
    """
    Extract feature matrix for a list of characters.
    Returns numpy array of shape (n_chars, n_features).

    Features per position i:
      - char type at i-w..i+w (categorical, 11 types)
      - is_thai at i-w..i+w
      - is_consonant at i-w..i+w
      - is_tone at i-w..i+w
      - is_lead_vowel at i-w..i+w
      - bigram / trigram type transitions
      - word boundary hint
    """
    n = len(chars)
    types  = [get_char_type(c) for c in chars]
    is_th  = [int(is_thai_char(c)) for c in chars]
    is_con = [int(c in THAI_CONSONANTS) for c in chars]
    is_ton = [int(c in THAI_TONEMARKS) for c in chars]
    is_lv  = [int(c in THAI_LEAD_VOWELS) for c in chars]
    is_fv  = [int(c in THAI_FOLL_VOWELS) for c in chars]
    is_dig = [int(c.isdigit() or c in THAI_DIGITS) for c in chars]
    is_alp = [int(c.isalpha()) for c in chars]
    is_pun = [int(not c.isalnum() and c not in ALL_THAI) for c in chars]

    feature_rows = []
    for i in range(n):
        row = []

        # Context window features
        for offset in range(-WINDOW, WINDOW + 1):
            j = i + offset
            pad = j < 0 or j >= n
            row += [
                types[j]  if not pad else -1,
                is_th[j]  if not pad else 0,
                is_con[j] if not pad else 0,
                is_ton[j] if not pad else 0,
                is_lv[j]  if not pad else 0,
                is_fv[j]  if not pad else 0,
                is_dig[j] if not pad else 0,
                is_alp[j] if not pad else 0,
                is_pun[j] if not pad else 0,
            ]

        # Bigram transition type: (type[i-1], type[i])
        prev_type = types[i - 1] if i > 0 else -1
        next_type = types[i + 1] if i < n - 1 else -1
        row.append(prev_type * 11 + types[i])  # bigram left
        row.append(types[i] * 11 + next_type)  # bigram right

        # Leading vowel before consonant pattern (E.g. เ+ก = เก)
        # If current is lead vowel AND next is consonant → likely boundary start
        if i < n - 1:
            row.append(int(chars[i] in THAI_LEAD_VOWELS and chars[i + 1] in THAI_CONSONANTS))
        else:
            row.append(0)

        # After tone mark → still within word
        if i > 0:
            row.append(int(chars[i - 1] in THAI_TONEMARKS))
        else:
            row.append(0)

        # Thai ↔ non-Thai script boundary
        row.append(is_word_boundary_hint(chars, i))

        feature_rows.append(row)

    return np.array(feature_rows, dtype=np.float32)


# Quick test
test_chars = list('สวัสดีครับ')
feat = extract_features(test_chars)
print(f'Feature shape for {len(test_chars)} chars: {feat.shape}')
print(f'Feature dim per char: {feat.shape[1]}')

Feature shape for 10 chars: (10, 104)
Feature dim per char: 104


In [7]:
# ── 4. Generate Training Data via PyThaiNLP Tokenizers ───────────────────────
# We use the test text + additional Thai text (from PyThaiNLP) as pseudo-labeled training
# Prime source: we tokenize many sentences and trust the ensemble intersection

# Use PyThaiNLP example texts + the test text for pseudo-labeled training
# The test text itself is a good proxy since it's news domain text

print('Generating pseudo-labeled training data from test text...')

# We'll use newmm (more accurate) as primary labels
# Split test text into chunks to avoid memory issues
CHUNK_SIZE = 1000  # chars per chunk

def chunk_text(text, chunk_size=1000):
    """Split text at whitespace boundaries into chunks."""
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        # Try to break at a whitespace
        if end < len(text):
            # Find nearest space
            space_pos = text.rfind(' ', start, end)
            if space_pos > start:
                end = space_pos + 1
        chunks.append(text[start:end])
        start = end
    return chunks


# Generate labels from both engines and take majority
def gen_ensemble_labels(text, engines=('newmm', 'longest')):
    """Return label arrays from multiple engines, then take majority vote."""
    label_matrices = {}  # engine → list of labels
    chars_ref = None

    for engine in engines:
        try:
            chars_e, labels_e = text_to_bio_labels(text, engine=engine)
            if chars_ref is None:
                chars_ref = chars_e
            if len(chars_e) == len(chars_ref):
                label_matrices[engine] = labels_e
        except Exception as ex:
            pass  # skip failed engines

    if not label_matrices:
        return None, None

    # Majority vote across engines at each position
    n = len(chars_ref)
    all_labels = list(label_matrices.values())
    voted = []
    for i in range(n):
        votes = [lbl[i] for lbl in all_labels]
        voted.append(Counter(votes).most_common(1)[0][0])

    return chars_ref, voted


# Process the full test text in chunks to create training data
chunks = chunk_text(test_text, CHUNK_SIZE)
print(f'Split into {len(chunks)} chunks')

all_train_chars = []
all_train_labels = []

for i, chunk in enumerate(chunks):
    chars_c, labels_c = gen_ensemble_labels(chunk)
    if chars_c:
        all_train_chars.extend(chars_c)
        all_train_labels.extend(labels_c)
    if (i + 1) % 20 == 0:
        print(f'  Processed {i+1}/{len(chunks)} chunks...')

print(f'\nTotal pseudo-labeled training chars: {len(all_train_chars):,}')
print('Label distribution:')
for k, v in Counter(all_train_labels).items():
    print(f'  {k}: {v:,} ({v/len(all_train_labels)*100:.1f}%)')

Generating pseudo-labeled training data from test text...
Split into 38 chunks
  Processed 20/38 chunks...

Total pseudo-labeled training chars: 35,194
Label distribution:
  B_WORD: 7,443 (21.1%)
  I_WORD: 20,637 (58.6%)
  E_WORD: 7,114 (20.2%)


In [8]:
# ── 5. Extract Features for Training ─────────────────────────────────────────
print('Extracting training features...')
X_train = extract_features(all_train_chars)

LABEL2INT = {'B_WORD': 0, 'I_WORD': 1, 'E_WORD': 2}
INT2LABEL = {v: k for k, v in LABEL2INT.items()}

y_train = np.array([LABEL2INT[l] for l in all_train_labels], dtype=np.int32)

print(f'X_train shape: {X_train.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'Class distribution: B={np.sum(y_train==0):,}  I={np.sum(y_train==1):,}  E={np.sum(y_train==2):,}')

Extracting training features...
X_train shape: (35194, 104)
y_train shape: (35194,)
Class distribution: B=7,443  I=20,637  E=7,114


In [9]:
# ── 6. Train LightGBM Classifier ──────────────────────────────────────────────
print('Training LightGBM classifier...')

lgb_params = {
    'objective':        'multiclass',
    'num_class':        3,
    'metric':           'multi_logloss',
    'learning_rate':    0.1,
    'num_leaves':       127,
    'max_depth':        -1,
    'min_child_samples': 20,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq':     5,
    'n_estimators':     500,
    'class_weight':     'balanced',
    'n_jobs':           -1,
    'verbose':          -1,
    'random_state':     42,
}

model = lgb.LGBMClassifier(**lgb_params)
model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)]
)
print(f'✅ Training complete. Best iteration: {model.best_iteration_}')

Training LightGBM classifier...
[100]	training's multi_logloss: 0.309723
[200]	training's multi_logloss: 0.234366
[300]	training's multi_logloss: 0.191952
[400]	training's multi_logloss: 0.164372
[500]	training's multi_logloss: 0.145694
✅ Training complete. Best iteration: 0


In [10]:
# ── 7. Self-Evaluation on Training Data ──────────────────────────────────────
y_pred_train = model.predict(X_train)
f1_macro = f1_score(y_train, y_pred_train, average='macro')
print(f'Training Macro F1: {f1_macro:.4f}  (this is optimistic / in-sample)')

label_names = ['B_WORD', 'I_WORD', 'E_WORD']
print(classification_report(y_train, y_pred_train, target_names=label_names))

Training Macro F1: 0.9263  (this is optimistic / in-sample)
              precision    recall  f1-score   support

      B_WORD       0.87      0.97      0.92      7443
      I_WORD       0.99      0.90      0.94     20637
      E_WORD       0.87      0.98      0.92      7114

    accuracy                           0.93     35194
   macro avg       0.91      0.95      0.93     35194
weighted avg       0.94      0.93      0.93     35194



In [11]:
# ── 8. Extract Features for Test Set ─────────────────────────────────────────
print('Extracting test set features...')
X_test = extract_features(non_ws_chars)
print(f'X_test shape: {X_test.shape}')

Extracting test set features...
X_test shape: (35182, 104)


In [ ]:
# ── 9. Multi-Strategy Ensemble Prediction ────────────────────────────────────
# Strategy: combine LightGBM probs + PyThaiNLP tokenizer votes

print('=== Strategy A: LightGBM predictions ===')
lgb_probs = model.predict_proba(X_test)  # shape (n, 3)  [B, I, E]
lgb_preds = np.argmax(lgb_probs, axis=1)

print('\n=== Strategy B: Multi-engine PyThaiNLP tokenizer ensemble ===')

# Apply tokenizers directly to test text
def get_tokenizer_probs(text, non_ws_chars, engines):
    """Run engines on text, return soft prob matrix over [B, I, E] for each non-ws char."""
    n = len(non_ws_chars)
    prob_accum = np.zeros((n, 3), dtype=np.float32)
    valid_engines = 0

    for engine in engines:
        try:
            chars_e, labels_e = gen_ensemble_labels(text, engines=[engine])
            if chars_e and len(chars_e) == n:
                for i, lbl in enumerate(labels_e):
                    prob_accum[i, LABEL2INT[lbl]] += 1.0
                valid_engines += 1
        except Exception as e:
            print(f'  {engine} failed: {e}')

    if valid_engines > 0:
        prob_accum /= valid_engines
    return prob_accum


# Run all text through tokenizers in chunks
print('Running newmm tokenizer on test text...')
_, newmm_labels = gen_ensemble_labels(test_text, engines=['newmm'])
print('Running longest tokenizer on test text...')
_, longest_labels = gen_ensemble_labels(test_text, engines=['longest'])

# Build soft probability matrices from tokenizer outputs
tok_probs = np.zeros((len(non_ws_chars), 3), dtype=np.float32)
for i, (l1, l2) in enumerate(zip(newmm_labels or ['B_WORD']*len(non_ws_chars),
                                   longest_labels or ['B_WORD']*len(non_ws_chars))):
    tok_probs[i, LABEL2INT[l1]] += 0.5
    tok_probs[i, LABEL2INT[l2]] += 0.5

print(f'\nTokenizer label distribution (newmm):')
if newmm_labels:
    for k, v in Counter(newmm_labels).items():
        print(f'  {k}: {v:,}')
    print(f'Tokenizer label distribution (longest):')
    for k, v in Counter(longest_labels).items():
        print(f'  {k}: {v:,}')

=== Strategy A: LightGBM predictions ===

=== Strategy B: Multi-engine PyThaiNLP tokenizer ensemble ===
Running newmm tokenizer on test text...
Running longest tokenizer on test text...


In [ ]:
# ── 10. Weighted Ensemble & Post-processing ───────────────────────────────────
# Combine LightGBM (trained on pseudo-labels) with PyThaiNLP tokenizer outputs
# Weight: 60% tokenizer ensemble (most reliable baseline), 40% LightGBM (for refinement)

W_TOK = 0.60
W_LGB = 0.40

ensemble_probs = W_TOK * tok_probs + W_LGB * lgb_probs
raw_preds = np.argmax(ensemble_probs, axis=1)
raw_pred_labels = [INT2LABEL[p] for p in raw_preds]

print(f'Raw ensemble predictions:')
for k, v in Counter(raw_pred_labels).items():
    print(f'  {k}: {v:,} ({v/len(raw_pred_labels)*100:.1f}%)')


# ── Post-processing: fix invalid BIO sequences ────────────────────────────────
def fix_bio_sequence(labels):
    """
    Fix invalid transitions in BIO label sequence:
      - I_WORD not preceded by B_WORD or I_WORD → change to B_WORD
      - E_WORD not preceded by B_WORD or I_WORD → change to B_WORD
      - B_WORD immediately before E_WORD is fine (2-char word)
      - Single B_WORD with no following I or E is fine (1-char word)
    """
    fixed = list(labels)
    n = len(fixed)

    for i in range(n):
        lbl = fixed[i]
        if i == 0:
            # First label must be B_WORD
            if lbl != 'B_WORD':
                fixed[i] = 'B_WORD'
            continue

        prev = fixed[i - 1]
        if lbl == 'I_WORD':
            # I_WORD must follow B_WORD or I_WORD
            if prev not in ('B_WORD', 'I_WORD'):
                fixed[i] = 'B_WORD'
        elif lbl == 'E_WORD':
            # E_WORD must follow B_WORD or I_WORD
            if prev not in ('B_WORD', 'I_WORD'):
                fixed[i] = 'B_WORD'
        # B_WORD can follow anything

    return fixed


final_labels = fix_bio_sequence(raw_pred_labels)

print(f'\nFinal predictions after BIO fix:')
for k, v in Counter(final_labels).items():
    print(f'  {k}: {v:,} ({v/len(final_labels)*100:.1f}%)')

In [ ]:
# ── 11. Validate & Sanity Check ───────────────────────────────────────────────
print(f'Submission rows: {len(final_labels):,}  (expected 35,182)')
assert len(final_labels) == 35182, f'❌ Expected 35182 rows, got {len(final_labels)}'
assert all(l in ('B_WORD', 'I_WORD', 'E_WORD') for l in final_labels), '❌ Invalid labels found!'

# Check BIO integrity
violations = 0
for i in range(1, len(final_labels)):
    if final_labels[i] in ('I_WORD', 'E_WORD') and final_labels[i-1] == 'E_WORD':
        violations += 1
print(f'BIO violations after fix: {violations}')

# Show a sample of predictions
print('\nSample predictions (chars 1..30):')
print(f'{"ID":>6}  {"Char":>6}  {"Label":>8}')
print('-' * 28)
for i in range(30):
    print(f'{non_ws_ids[i]:>6}  {repr(non_ws_chars[i]):>6}  {final_labels[i]:>8}')

print('\n✅ All validation checks passed!')

In [ ]:
# ── 12. Generate Submission File ──────────────────────────────────────────────
submission_df = pd.DataFrame({
    'Id': non_ws_ids,
    'Predicted': final_labels
})

submission_df.to_csv(OUTPUT_FILE, index=False)

print(f'✅ Submission saved to: {OUTPUT_FILE}')
print(f'   Rows: {len(submission_df):,}')
print(f'   Columns: {list(submission_df.columns)}')
print('\nHead:')
print(submission_df.head(10).to_string(index=False))
print('\nTail:')
print(submission_df.tail(5).to_string(index=False))
print('\nLabel distribution:')
print(submission_df['Predicted'].value_counts())

In [ ]:
# ── 13. Cross-Engine Agreement Analysis ──────────────────────────────────────
# Measure how much newmm and longest agree — high agreement = high confidence
if newmm_labels and longest_labels:
    agreements = sum(1 for a, b in zip(newmm_labels, longest_labels) if a == b)
    total = len(newmm_labels)
    print(f'Engine agreement (newmm vs longest): {agreements/total*100:.1f}% ({agreements:,}/{total:,})')

    disagreement_labels = [
        (nm, lo) for nm, lo in zip(newmm_labels, longest_labels) if nm != lo
    ]
    print('\nDisagreement patterns (top 10):')
    for pattern, count in Counter(disagreement_labels).most_common(10):
        print(f'  {pattern[0]:>8} vs {pattern[1]:>8}: {count:,}')

    # Compute F1 treating newmm as ground truth, show longest vs newmm
    nm_int = [LABEL2INT[l] for l in newmm_labels]
    lo_int = [LABEL2INT[l] for l in longest_labels]
    f1_between = f1_score(nm_int, lo_int, average='macro')
    print(f'\nF1 (longest vs newmm): {f1_between:.4f}')
    print('  (Indicates inter-engine consistency; higher = both agree more)')

In [ ]:
# ── 14. Summary ───────────────────────────────────────────────────────────────
print('=' * 60)
print('THAI WORD SEGMENTATION — SUBMISSION SUMMARY')
print('=' * 60)
print(f'Output file:   {OUTPUT_FILE}')
print(f'Total rows:    {len(submission_df):,}')
print(f'Label counts:')
for label, count in submission_df['Predicted'].value_counts().items():
    print(f'  {label:>10}: {count:>7,} ({count/len(submission_df)*100:.1f}%)')
print()
print('Method: Multi-engine PyThaiNLP ensemble (newmm + longest)')
print('        + LightGBM character feature classifier')
print('        + BIO sequence post-processing fix')
print('=' * 60)